# EXTERNAL iSign T5 — Kaggle Dataset-Level Evaluation (t5-small-motion ONLY)

Evaluation-only notebook. No training, no fine-tuning, no registry changes, no promotion, no frontend integration.

**Model:** `manavdhamecha77/iSign-t5-pose-to-text` checkpoint `t5-small-motion` ONLY (pose_dim=600, motion=True).
**Dataset:** held-out iSign test split (target 1,887 samples; actual verified count is recorded).
**Preprocessing:** verified external iSign contract (75 landmarks, +confidence 1.0, z-score axis=0 before motion, velocity concat, pad/truncate 500).
**Rule:** a prediction counts as correct ONLY if normalized prediction == normalized ground truth. Fluent English alone is never counted as correct.

Hard constraints (must hold):
- Do NOT modify the production SignSage inference pipeline.
- Do NOT train or fine-tune anything (`model.eval()` + `torch.no_grad()` only).
- Do NOT modify the model registry; do NOT set `SIGNSAGE_MODEL_PROMOTED=true`.
- Do NOT use `t5-base` / `t5-large` or any other checkpoint. ONLY `t5-small-motion`.
- Do NOT save hidden states, tensors, checkpoints, videos, images, dataset copies, or huge logs. Compact artifacts only (<10 MB).
- If the dataset cannot be accessed or its format is incompatible, STOP with an explicit blocker (no workaround, no invented numbers).
- If labels cannot be reliably mapped, STOP and document (no misleading accuracy).

Outputs (`evaluation_output/`): `predictions.csv`, `metrics.json`, `confusion_matrix.csv`, `evaluation_report.md`.

In [ ]:
import os
from pathlib import Path

# ---- All paths / knobs are configurable via environment; Kaggle defaults shown ----
CFG = {
    "MODEL_REPO": os.environ.get("ISIGN_MODEL_REPO", "manavdhamecha77/iSign-t5-pose-to-text"),
    "MODEL_VARIANT": os.environ.get("ISIGN_MODEL_VARIANT", "t5-small-motion"),
    "MODEL_DIR": os.environ.get("ISIGN_MODEL_DIR", ""),  # optional local/Kaggle-input checkpoint dir
    "BASE_TOKENIZER": os.environ.get("ISIGN_BASE_TOKENIZER", "t5-small"),
    "ISIGN_CSV": os.environ.get("ISIGN_CSV", ""),  # optional explicit CSV path
    "POSE_SOURCE": os.environ.get("ISIGN_POSE_SOURCE", ""),  # dir | zip | remote; empty = auto
    "POSE_KIND": os.environ.get("ISIGN_POSE_KIND", "auto"),  # auto | dir | zip | remote
    "KAGGLE_INPUT_ROOT": os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"),
    "KAGGLE_WORKING": os.environ.get("KAGGLE_WORKING", "/kaggle/working"),
    "OUTPUT_DIR": os.environ.get("EVAL_OUTPUT_DIR", "/kaggle/working/evaluation_output"),
    "MAX_EVAL_SAMPLES": int(os.environ.get("MAX_EVAL_SAMPLES", "1887")),
    "MAX_FRAMES": 500,
    "POSE_DIM": 600,
    "USE_MOTION": True,
    "MAX_GEN_TOKENS": 128,
    "NUM_BEAMS": 4,
    "SPLIT_SEED": 42,
}
assert CFG["MODEL_VARIANT"] == "t5-small-motion", "ONLY t5-small-motion is allowed for this evaluation"
assert CFG["POSE_DIM"] == 600 and CFG["USE_MOTION"] is True, "t5-small-motion contract: pose_dim=600 + motion"
print("== Configuration summary ==")
print(f"Model: {CFG['MODEL_VARIANT']} (repo {CFG['MODEL_REPO']})")
print(f"Dataset: iSign held-out test set (target min({CFG['MAX_EVAL_SAMPLES']}, 1887); actual count recorded at runtime)")
print(f"Preprocessing: verified external iSign preprocessing (75 landmarks, z-score axis=0, motion, pad/truncate {CFG['MAX_FRAMES']})")
print(f"Generation: max_tokens={CFG['MAX_GEN_TOKENS']} num_beams={CFG['NUM_BEAMS']} (verified contract)")
print(f"Output dir: {CFG['OUTPUT_DIR']} (configurable via EVAL_OUTPUT_DIR)")


In [ ]:
import platform, sys
print(f"python={platform.python_version()} executable={sys.executable}")
try:
    import torch
    print(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"gpu={torch.cuda.get_device_name(0)} count={torch.cuda.device_count()}")
    else:
        print("device fallback: CPU")
except Exception as e:
    print(f"torch import failed pre-install: {type(e).__name__}: {e}")
try:
    import transformers
    print(f"transformers={transformers.__version__}")
except Exception as e:
    print(f"transformers not yet installed: {type(e).__name__}")
try:
    import pose_format
    print("pose_format=available")
except Exception:
    print("pose_format=missing (will install)")
import pandas as _pd, numpy as _np
print(f"pandas={_pd.__version__} numpy={_np.__version__}")


In [ ]:
import subprocess, sys
# Minimal eval-only dependencies. torch/numpy/pandas are preinstalled on Kaggle images.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "huggingface_hub", "pose-format", "pandas"], check=True)
import torch, transformers
print(f"installed: torch={torch.__version__} transformers={transformers.__version__}")


In [ ]:
import random
import numpy as np, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} (GPU if available, otherwise CPU)")
torch.set_grad_enabled(False)


## Verified external preprocessing contract (read-only, eval-only)

Sources: `docs/EXTERNAL_ISIGN_CONTRACT_VERIFICATION.md`, `docs/EXTERNAL_ISIGN_ADAPTER.md`,
`backend/app/ml/adapters/i_sign_t5_adapter.py` (`signage_to_external`), external `inference/pose_utils.py` + `inference/model.py`.

| Step | Exact rule |
|---|---| 
| 1. Component select | SignSage 543 order POSE(0:33)+FACE(33:501)+LEFT(501:522)+RIGHT(522:543); keep POSE + LEFT + RIGHT = 75 landmarks; discard FACE |
| 2. Confidence | append 1.0 per landmark -> 75 x 4 = 300 values/frame (`include_confidence=True` fallback) |
| 3. Normalize | z-score across axis=0: `(x - nanmean(x,0)) / where(nanstd==0, 1.0, nanstd)`, then `nan_to_num` |
| 4. Motion | FROM normalized features: `velocity[0]=0`, `velocity[t]=pose[t]-pose[t-1]`; concat `[pose \| velocity]` -> 600/frame |
| 5. Sequence | pad with zeros / truncate to exactly 500 frames; `actual_length = min(T, 500)` |
| 6. Tensors | `pose_tensor=(1,500,600)` float32, `pose_length=[actual_length]` long; attention mask from `actual_length` |

Order matters: normalize BEFORE motion, motion BEFORE frame padding. No SignSage hip-centering here.

In [ ]:
import numpy as np

TOTAL_LANDMARKS = 543
POSE_SL = slice(0, 33)
LEFT_SL = slice(501, 522)
RIGHT_SL = slice(522, 543)
SPATIAL_DIM, MOTION_DIM, MAX_FRAMES = 300, 600, 500

def normalize_text(text) -> str:
    """Lowercase + collapse whitespace (same as research/training/audit_dataset.py)."""
    return " ".join(str(text).lower().strip().split())

def signage_to_external_verified(landmarks_543: np.ndarray, use_motion: bool = True, max_frames: int = 500):
    """Verbatim port of backend/app/ml/adapters/i_sign_t5_adapter.py::signage_to_external.
    Input (T,543,3) -> (max_frames,300|600) float32 + actual_length. Eval-only, no training."""
    if not isinstance(landmarks_543, np.ndarray):
        raise TypeError("Expected numpy.ndarray (T,543,3)")
    if landmarks_543.ndim != 3 or landmarks_543.shape[1] != 543 or landmarks_543.shape[2] != 3:
        raise ValueError(f"Expected (T,543,3), got {landmarks_543.shape} (incompatible representation -- refusing)")
    T = int(landmarks_543.shape[0])
    if T < 1:
        raise ValueError("Empty pose sequence (incompatible -- refusing)")
    pose_33 = landmarks_543[:, POSE_SL, :]
    left_21 = landmarks_543[:, LEFT_SL, :]
    right_21 = landmarks_543[:, RIGHT_SL, :]
    parts = []
    for comp in (pose_33, left_21, right_21):
        flat3 = comp.reshape(T, -1, 3).astype(np.float32)
        conf = np.ones((T, flat3.shape[1], 1), dtype=np.float32)  # documented fallback 1.0
        parts.append(np.concatenate([flat3, conf], axis=2).reshape(T, -1))
    combined = np.concatenate(parts, axis=1).astype(np.float32)  # (T,300)
    assert combined.shape[1] == SPATIAL_DIM, combined.shape
    mean = np.nanmean(combined, axis=0, keepdims=True)
    std = np.nanstd(combined, axis=0, keepdims=True)
    combined = (combined - mean) / np.where(std == 0, 1.0, std)
    combined = np.nan_to_num(combined, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    if use_motion:
        velocity = np.zeros_like(combined, dtype=np.float32)
        velocity[1:] = combined[1:] - combined[:-1]
        combined = np.concatenate([combined, velocity], axis=1).astype(np.float32)  # (T,600)
        assert combined.shape[1] == MOTION_DIM, combined.shape
    if T >= max_frames:
        return combined[:max_frames].copy(), max_frames
    padded = np.zeros((max_frames, combined.shape[1]), dtype=np.float32)
    padded[:T] = combined
    return padded, T

def read_pose_to_543(raw_bytes: bytes, uid: str) -> np.ndarray:
    """Raw .pose bytes -> (T,543,3) float32. Verifies component layout before positional slicing."""
    from pose_format import Pose
    pose = Pose.read(raw_bytes)
    names = [c.name for c in pose.header.components]
    expected_prefix = ["POSE_LANDMARKS", "FACE_LANDMARKS", "LEFT_HAND_LANDMARKS", "RIGHT_HAND_LANDMARKS"]
    if names[:4] != expected_prefix:
        raise ValueError(f"Pose header {names} for {uid} does not start with {expected_prefix} -- refusing to slice positionally")
    total = sum(len(c.points) for c in pose.header.components)
    if total < TOTAL_LANDMARKS:
        raise ValueError(f"Pose header has only {total} points for {uid}, need {TOTAL_LANDMARKS}")
    data = np.asarray(np.ma.filled(pose.body.data, 0.0), dtype=np.float32)
    if data.ndim != 4 or data.shape[1] < 1:
        raise ValueError(f"Unexpected pose tensor {data.shape} for {uid}")
    return data[:, 0, :TOTAL_LANDMARKS, :3].astype(np.float32)  # (T,543,3)

# Self-check on synthetic data (shapes + motion formula only; NOT an accuracy claim).
_rng = np.random.default_rng(0)
_p, _l = signage_to_external_verified(_rng.random((7, 543, 3), dtype=np.float32), use_motion=True, max_frames=500)
assert _p.shape == (500, 600) and _l == 7 and np.isfinite(_p).all()
print("preprocessing self-check OK: (7,543,3) -> (500,600), length=7")


In [ ]:
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from huggingface_hub import snapshot_download
from pathlib import Path

class PoseEncoder(torch.nn.Module):
    """Verified arch from inference/model.py: Linear(pose_dim->256)->ReLU->Dropout(0.1)->Linear(256->d_model)."""
    def __init__(self, pose_dim: int, embedding_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(pose_dim, hidden_dim), torch.nn.ReLU(),
            torch.nn.Dropout(0.1), torch.nn.Linear(hidden_dim, embedding_dim))
    def forward(self, pose):
        return self.encoder(pose)

def resolve_checkpoint():
    if CFG["MODEL_DIR"] and Path(CFG["MODEL_DIR"]).is_dir():
        ckpt = Path(CFG["MODEL_DIR"])
        print(f"using local checkpoint dir: {ckpt}")
    else:
        print(f"downloading {CFG['MODEL_REPO']} subfolder {CFG['MODEL_VARIANT']} (compact: t5-small-motion only)")
        root = snapshot_download(repo_id=CFG["MODEL_REPO"], allow_patterns=[f"{CFG['MODEL_VARIANT']}/*"])
        ckpt = Path(root) / CFG["MODEL_VARIANT"]
    t5_dir, pe_path = ckpt / "t5", ckpt / "pose_encoder.pt"
    if not t5_dir.is_dir() or not pe_path.is_file():
        raise RuntimeError(f"checkpoint incomplete at {ckpt}: need t5/ + pose_encoder.pt")
    return ckpt, t5_dir, pe_path

CKPT_DIR, T5_DIR, PE_PATH = resolve_checkpoint()
print(f"checkpoint: {CFG['MODEL_REPO']}:{CFG['MODEL_VARIANT']} @ {CKPT_DIR}")
tokenizer = AutoTokenizer.from_pretrained(CFG["BASE_TOKENIZER"])
t5_model = T5ForConditionalGeneration.from_pretrained(str(T5_DIR)).to(DEVICE)
t5_model.eval()
pose_encoder = PoseEncoder(pose_dim=600, embedding_dim=t5_model.config.d_model).to(DEVICE)
pose_encoder.load_state_dict(torch.load(str(PE_PATH), map_location=DEVICE))
pose_encoder.eval()
assert t5_model.training is False and pose_encoder.training is False
print(f"loaded t5-small-motion: d_model={t5_model.config.d_model} device={DEVICE} (eval-only, no training)")


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

def clean_dataframe_lite(df: pd.DataFrame):
    """Minimal port of research/training/audit_dataset.py::clean_dataframe (uid/text only)."""
    df = df.copy()
    df["uid"] = df["uid"].astype(str).str.strip()
    df["text"] = df["text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    parts = df["uid"].str.rsplit("-", n=1, expand=True)
    df["video_id"] = parts[0].where(parts[1].notna() & (parts[0].str.strip() != ""), None)
    df = df.loc[df["video_id"].notna()].copy()
    df["text_norm"] = df["text"].map(normalize_text)
    junk = df["text_norm"].eq("") | df["uid"].eq("")
    return df.loc[~junk].reset_index(drop=True)

def video_split_lite(df: pd.DataFrame, seed: int = 42):
    """Leak-safe 80/10/10 split by video_id (same policy as audit_dataset.video_split)."""
    vids = np.array(sorted(df["video_id"].dropna().unique()), dtype=object)
    rng = np.random.default_rng(seed)
    rng.shuffle(vids)
    n = len(vids)
    c1, c2 = int(0.80 * n), int(0.90 * n)
    tr_ids, va_ids, te_ids = set(vids[:c1]), set(vids[c1:c2]), set(vids[c2:])
    te = df[df["video_id"].isin(te_ids)].reset_index(drop=True)
    return te

def find_csv():
    if CFG["ISIGN_CSV"] and Path(CFG["ISIGN_CSV"]).is_file():
        return CFG["ISIGN_CSV"]
    for pat in ("**/iSign_v1.1.csv", "**/isign*.csv", "**/*.csv"):
        hits = sorted(Path(CFG["KAGGLE_INPUT_ROOT"]).glob(pat)) if Path(CFG["KAGGLE_INPUT_ROOT"]).is_dir() else []
        for h in hits:
            try:
                if {"uid", "text"} <= set(pd.read_csv(h, nrows=0).columns):
                    print(f"found CSV under Kaggle input: {h}")
                    return str(h)
            except Exception:
                continue
    # Small metadata CSV fallback from the gated HF dataset (needs HF_TOKEN secret).
    import os
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = None
    if not tok:
        raise RuntimeError("BLOCKER: no iSign CSV found (set ISIGN_CSV or attach it under /kaggle/input) and no HF_TOKEN secret for HF fallback. Stopping -- no workaround.")
    from huggingface_hub import HfApi, hf_hub_download
    cands = [f for f in HfApi(token=tok).list_repo_files("Exploration-Lab/iSign", repo_type="dataset") if f.endswith(".csv")]
    pick = next((f for f in cands if f.lower().endswith("isign_v1.1.csv")), None) or (cands[0] if len(cands) == 1 else None)
    if pick is None:
        raise RuntimeError(f"BLOCKER: cannot choose iSign CSV on HF; candidates={cands}. Stopping.")
    path = hf_hub_download("Exploration-Lab/iSign", pick, repo_type="dataset", token=tok)
    print(f"downloaded CSV {pick}")
    return path

CSV_PATH = find_csv()
df_all = pd.read_csv(CSV_PATH)
if not {"uid", "text"} <= set(df_all.columns):
    raise RuntimeError(f"BLOCKER: CSV {CSV_PATH} header {list(df_all.columns)} missing uid/text (incompatible -- stopping).")
df_clean = clean_dataframe_lite(df_all)
df_test_full = video_split_lite(df_clean, seed=CFG["SPLIT_SEED"])
N_TARGET = min(int(CFG["MAX_EVAL_SAMPLES"]), 1887, len(df_test_full))
df_test = df_test_full.iloc[:N_TARGET].reset_index(drop=True)
print(f"CSV: {CSV_PATH} rows={len(df_all)} usable={len(df_clean)} test_pool={len(df_test_full)} evaluating={len(df_test)}")
if len(df_test) == 0:
    raise RuntimeError("BLOCKER: held-out test split is empty after cleaning (stopping, no invented labels).")


In [ ]:
# Pose-source resolution: local dir | local zip | HF remote streaming (range requests, no full download).
import zipfile
from pathlib import Path

class DirPoseSource:
    def __init__(self, root): self.root = Path(root)
    def read(self, uid):
        p = self.root / f"{uid}.pose"
        if not p.is_file(): raise FileNotFoundError(uid)
        return p.read_bytes()

class ZipPoseSource:
    def __init__(self, path):
        self.zf = zipfile.ZipFile(path, allowZip64=True); self.names = set(self.zf.namelist())
    def read(self, uid):
        for cand in (f"iSign-poses_v1.1/{uid}.pose", f"{uid}.pose"):
            if cand in self.names:
                with self.zf.open(cand) as fh: return fh.read()
        raise FileNotFoundError(uid)

def find_pose_source():
    if CFG["POSE_SOURCE"]:
        s = CFG["POSE_SOURCE"]
        if zipfile.is_zipfile(s): return ("zip", ZipPoseSource(s))
        if Path(s).is_dir(): return ("dir", DirPoseSource(s))
        if s == "remote": return ("remote", None)
        raise RuntimeError(f"BLOCKER: ISIGN_POSE_SOURCE={s!r} is not a dir/zip/remote (stopping).")
    root = Path(CFG["KAGGLE_INPUT_ROOT"])
    if root.is_dir():
        zips = sorted(root.rglob("*.zip")) + sorted(root.rglob("*.pose"))
        for z in zips:
            if z.suffix == ".zip" and zipfile.is_zipfile(z):
                print(f"found pose zip: {z}")
                return ("zip", ZipPoseSource(str(z)))
        dirs = [d for d in root.rglob("*") if d.is_dir() and next(d.glob("*.pose"), None)]
        if dirs:
            print(f"found pose dir: {dirs[0]}")
            return ("dir", DirPoseSource(str(dirs[0])))
    return ("remote", None)

POSE_KIND, POSE_STORE = find_pose_source()
print(f"pose source kind: {POSE_KIND}")
if POSE_KIND == "remote":
    raise RuntimeError("BLOCKER: no local pose dir/zip attached. Attach iSign poses as a Kaggle input dataset, or set ISIGN_POSE_SOURCE to a dir/zip/remote endpoint. Remote streaming needs HF_TOKEN + range access and is not auto-enabled here to avoid large hidden downloads. Stopping -- no synthetic workaround for accuracy.")
# Spot-check ONE member before the full loop (fail fast on incompatible layout).
_probe_uid = str(df_test["uid"].iloc[0])
_raw = POSE_STORE.read(_probe_uid)
_arr = read_pose_to_543(_raw, _probe_uid)
print(f"pose preflight OK: uid={_probe_uid} frames={_arr.shape[0]} shape={_arr.shape}")


In [ ]:
import time, torch
from collections import Counter

@torch.no_grad()
def generate_text(pose_600: torch.Tensor, actual_len: int) -> str:
    pose_600 = pose_600.to(DEVICE)
    emb = pose_encoder(pose_600)  # (1,500,512)
    L = emb.size(1)
    mask = (torch.arange(L, device=DEVICE)[None, :] < torch.tensor([actual_len], device=DEVICE)[:, None]).long()
    gen = t5_model.generate(inputs_embeds=emb, attention_mask=mask, max_length=CFG["MAX_GEN_TOKENS"],
                            num_beams=CFG["NUM_BEAMS"], early_stopping=True, length_penalty=2.0)
    return tokenizer.decode(gen[0], skip_special_tokens=True)

rows, total_pre, total_gen, t0_all = [], 0.0, 0.0, time.time()
N = len(df_test)
for i, r in df_test.iterrows():
    uid, expected_raw = str(r["uid"]), str(r["text"])
    expected_norm = normalize_text(expected_raw)
    t0 = time.time()
    raw = POSE_STORE.read(uid)
    arr543 = read_pose_to_543(raw, uid)
    n_frames = int(arr543.shape[0])
    pre, actual_len = signage_to_external_verified(arr543, use_motion=True, max_frames=500)
    t1 = time.time()
    pt = torch.from_numpy(pre).unsqueeze(0).float()
    pred_raw = generate_text(pt, int(actual_len))
    t2 = time.time()
    pred_norm = normalize_text(pred_raw)
    exact = bool(expected_norm) and (pred_norm == expected_norm)
    rows.append({"sample_id": uid, "expected_raw": expected_raw, "expected_norm": expected_norm,
                 "predicted_raw": pred_raw, "predicted_norm": pred_norm, "exact_match": bool(exact),
                 "latency_s": round(t2 - t1, 4), "preprocess_s": round(t1 - t0, 4),
                 "num_frames": n_frames, "actual_length": int(actual_len)})
    total_pre += (t1 - t0); total_gen += (t2 - t1)
    if (i + 1) % 100 == 0 or (i + 1) == N:
        print(f"[{i+1}/{N}]")
total_elapsed = time.time() - t0_all
print(f"done: n={N} preprocessing={total_pre:.1f}s inference={total_gen:.1f}s total_elapsed={total_elapsed:.1f}s")


In [ ]:
import json, statistics
from collections import Counter
from pathlib import Path
import pandas as pd

OUT = Path(CFG["OUTPUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
pred = pd.DataFrame(rows)
n = len(pred)
n_ok = int(pred["exact_match"].sum()); n_bad = n - n_ok
acc = (n_ok / n) if n else 0.0
lat = pred["latency_s"].tolist()
metrics = {
    "model_repo": CFG["MODEL_REPO"], "checkpoint": CFG["MODEL_VARIANT"],
    "checkpoint_dir": str(CKPT_DIR), "tokenizer": CFG["BASE_TOKENIZER"],
    "dataset": "iSign held-out test set (video-level split, seed 42)", "csv": str(CSV_PATH),
    "n_samples": n, "exact_match_accuracy": acc, "n_correct": n_ok, "n_incorrect": n_bad,
    "avg_latency_s": float(sum(lat) / len(lat)) if lat else 0.0,
    "median_latency_s": float(statistics.median(lat)) if lat else 0.0,
    "total_inference_s": float(sum(lat)), "total_preprocessing_s": round(total_pre, 3),
    "total_elapsed_s": round(total_elapsed, 3), "device": str(DEVICE),
    "preprocessing": "verified external iSign (75 landmarks, conf 1.0, z-score axis=0, motion, pad/truncate 500)",
    "generation": {"max_length": CFG["MAX_GEN_TOKENS"], "num_beams": CFG["NUM_BEAMS"]},
    "eval_rule": "correct ONLY if normalized prediction == normalized ground truth",
    "empty_prediction_rate": float((pred['predicted_norm'] == "").mean()) if n else 0.0,
}
pred.to_csv(OUT / "predictions.csv", index=False)
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
# Compact confusion: top expected->predicted pairs (sentence labels; full matrix infeasible).
pairs = Counter(zip(pred["expected_norm"], pred["predicted_norm"]))
conf = pd.DataFrame([{"expected_norm": e, "predicted_norm": p, "count": c}
                     for (e, p), c in pairs.most_common(50)])
conf.to_csv(OUT / "confusion_matrix.csv", index=False)
wrong = pred[~pred["exact_match"]]
print(f"n={n} acc={acc:.4f} correct={n_ok} incorrect={n_bad}")
print(f"latency avg={metrics['avg_latency_s']:.3f}s median={metrics['median_latency_s']:.3f}s total_gen={metrics['total_inference_s']:.1f}s")
print("most common incorrect predictions:")
for txt, c in Counter(wrong["predicted_norm"]).most_common(10):
    print(f"  {c}x {txt[:120]!r}")
print(f"saved: predictions.csv, metrics.json, confusion_matrix.csv -> {OUT}")


In [ ]:
from pathlib import Path
from collections import Counter

OUT = Path(CFG["OUTPUT_DIR"])
wrong = pred[~pred["exact_match"]]
top_wrong = Counter(wrong["predicted_norm"]).most_common(10)
top_pairs = Counter(zip(pred["expected_norm"], pred["predicted_norm"])).most_common(10)
total_mb = sum(p.stat().st_size for p in OUT.glob("*") if p.is_file()) / 1e6
report = f"""# Evaluation report — external iSign t5-small-motion (dataset-level)

## 1. What was evaluated
- Model: `{CFG['MODEL_REPO']}` checkpoint `{CFG['MODEL_VARIANT']}` ONLY (pose_dim=600, motion=True).
- Tokenizer: `{CFG['BASE_TOKENIZER']}`. Generation: max_length={CFG['MAX_GEN_TOKENS']}, beams={CFG['NUM_BEAMS']}.
- Dataset: iSign held-out test split (video-level 80/10/10, seed 42) from `{CSV_PATH}`; evaluated n={metrics['n_samples']}.
- Preprocessing: verified external contract ONLY (75 landmarks POSE+LEFT+RIGHT, confidence=1.0 fallback, z-score across axis=0 BEFORE motion, velocity[0]=0 + velocity[t]=pose[t]-pose[t-1], pad/truncate 500). No SignSage hip-centering, no preprocessing changes.
- Production pipeline untouched; registry untouched; `SIGNSAGE_MODEL_PROMOTED` untouched (no promotion).

## 2. What was actually measured
- Per-sample: sample ID, expected/ground-truth label, raw prediction, normalized prediction, exact-match (normalized equality ONLY), generation latency, preprocessing latency, frame counts.
- Aggregates: exact-match accuracy, correct/incorrect counts, avg/median/total inference latency, preprocessing/total elapsed time.
- Error analysis: top confused expected->predicted pairs (compact; sentence labels make a full matrix infeasible) + most common incorrect predictions + empty-output rate.
- Fluent English output was NEVER counted as correct unless it exactly matched ground truth under the defined rule.

## 3. Quantitative results
- Samples: {metrics['n_samples']}
- Exact-match accuracy: {metrics['exact_match_accuracy']:.4f} ({metrics['n_correct']}/{metrics['n_samples']})
- Incorrect: {metrics['n_incorrect']}
- Avg latency: {metrics['avg_latency_s']:.3f}s | median: {metrics['median_latency_s']:.3f}s | total inference: {metrics['total_inference_s']:.1f}s
- Preprocessing total: {metrics['total_preprocessing_s']:.1f}s | elapsed total: {metrics['total_elapsed_s']:.1f}s | device: {metrics['device']}
- Empty-prediction rate: {metrics['empty_prediction_rate']:.4f}

## 4. Common errors
"""
for txt, c in top_wrong:
    report += f"- incorrect prediction {c}x: {txt[:160]!r}\n"
report += "\nTop expected->predicted pairs:\n"
for (e, p), c in top_pairs:
    report += f"- {c}x expected={e[:100]!r} predicted={p[:100]!r}\n"
report += f"""
## 5. Limitations
- Sentence-level labels: no word-level confusion matrix; pairs above are the compact substitute.
- Confidence fallback 1.0 (SignSage/frontend provides no per-landmark confidence; training distribution unknown).
- Studio training data vs any webcam distribution shift is NOT measured here.
- Held-out split reconstructed via video-level policy (seed 42); if the publisher's exact 1,887 IDs differ, counts shift -- actual n is recorded above.
- Accuracy is exact-match only; near-paraphrases count as incorrect by design (no silent leniency).

## 6. Does this justify proceeding to real SignSage webcam testing?
- This notebook alone makes NO production-readiness claim.
- Proceed to controlled webcam testing ONLY as a separate, small-vocabulary experiment with hallucination/negative tests (empty frame, random movement must not yield fluent text).
- Keep `SIGNSAGE_MODEL_PROMOTED` unset until that webcam quality evaluation passes explicit GO criteria.

Artifacts: predictions.csv, metrics.json, confusion_matrix.csv, evaluation_report.md (total ~{total_mb:.2f} MB).
"""
(OUT / "evaluation_report.md").write_text(report)
print(f"wrote evaluation_report.md ({total_mb:.2f} MB total in {OUT})")
print("== Final config ==")
print(f"Model: {CFG['MODEL_VARIANT']} | Dataset: iSign held-out test | Samples: {metrics['n_samples']} | Preprocessing: verified external | Device: {metrics['device']}")
